In [4]:
import torch

In [5]:
def sample_domain():
    X = torch.tensor(
           [[0.9250, 0.9151, 0.32],
            [0.0815, 0.3014, 0.16],
            [0.2055, 0.2310, 0.67],
            [0.9597, 0.1719, 0.75]]
    )
    return X

r2 = lambda x: torch.sum(x**2, dim=1).unsqueeze(dim=1)
model = lambda x: torch.cos(r2(x))

X = sample_domain()
X.requires_grad_(True)
u = model(X)
print(X.shape, u.shape)

torch.Size([4, 3]) torch.Size([4, 1])


---

## SDGD

In [6]:
"""
loss_pde = torch.mean(residual**2) = 1/N [()**2 + ... + ()**2]

2*residual * 
"""

'\nloss_pde = torch.mean(residual**2) = 1/N [()**2 + ... + ()**2]\n\n2*residual * \n'

In [41]:
torch.randint(0,4, (5,3), dtype=torch.float)

tensor([[1., 1., 1.],
        [0., 2., 3.],
        [1., 2., 3.],
        [0., 1., 1.],
        [1., 3., 2.]])

In [7]:
def compute_derivatives(model, X):
    """
    Compute u, grad u, and laplace u
    X: (batch_size, D) where D = d + 1 (spatial dims + time)
    """
    u = model(X)
    bs, D = X.shape

    # Gradient - spatial & temporatal
    grad_u = torch.autograd.grad(
        inputs=X,
        outputs=u,
        grad_outputs=torch.ones_like(u),
        create_graph=True,
        retain_graph=True,
    )[0]

    # Laplacian - spatial only
    #spatial_laplace_u = torch.zeros((bs,D-1))
    spatial_laplace_u = []
    for i in range(D-1):
        hess_row = torch.autograd.grad(
            inputs=X,
            outputs=grad_u[:,i].sum(),
            grad_outputs=torch.tensor(1.0),
            create_graph=True,
            retain_graph=True
        )[0]
        spatial_laplace_u.append(hess_row[:,i:i+1])
    spatial_laplace_u = torch.cat(spatial_laplace_u, dim=1)

    # shapes: bs x 1, bs x D, bs x D-1
    #return u, grad_u, spatial_laplace_u
    return u, grad_u, spatial_laplace_u

In [8]:
u, grad_u, spatial_laplace_u = compute_derivatives(model, X)
spatial_laplace_u

tensor([[-1.1874, -1.2036],
        [-0.2719, -0.6062],
        [-1.1805, -1.2185],
        [-2.2092, -2.0035]], grad_fn=<CatBackward0>)

In [9]:
spatial_laplace_u.sum(dim=1)

tensor([-2.3910, -0.8781, -2.3990, -4.2127], grad_fn=<SumBackward1>)

In [19]:
spatial_laplace_u

tensor([[-1.1874, -1.2036],
        [-0.2719, -0.6062],
        [-1.1805, -1.2185],
        [-2.2092, -2.0035]], grad_fn=<CatBackward0>)

In [23]:
X = sample_domain()
X.requires_grad_(True)
N,D = X.shape
d = D-1
print(f"d = {d}")
u, grad_u, spatial_laplace_u = compute_derivatives(model, X)
laplace_tot = torch.zeros_like(u)
for i in range(d):
    print(i)
    laplace_tot += spatial_laplace_u[:,i:i+1]
# params
alpha = 1.0
b = 1.0
v = torch.ones(d,1)
# residual
R = grad_u[:,-1:] - alpha * laplace_tot + torch.matmul(grad_u[:,:-1], v) + b * u

# i-th dim residual
i = 1
Ri = 1/d * grad_u[:,-1:] - alpha * spatial_laplace_u[i] + v[i] * grad_u[:,i:i+1] + 1/d * b * u
# total loss
loss = 2 * R.detach() * R
loss = torch.mean(loss)
loss.backward()
print(f"X.grad = {X.grad}")


d = 2
0
1
X.grad = tensor([[17.8629, 17.6930,  6.8999],
        [ 0.7341,  3.2902,  0.5432],
        [ 0.7423,  0.9699,  0.0557],
        [-2.7546, -0.9142, -2.3536]])


In [25]:
X = sample_domain()
X.requires_grad_(True)
N,D = X.shape
d = D-1
print(f"d = {d}")
u, grad_u, spatial_laplace_u = compute_derivatives(model, X)
laplace_tot = torch.zeros_like(u)
for i in range(d):
    laplace_tot += spatial_laplace_u[:,i:i+1]
# params
alpha = 1.0
b = 1.0
v = torch.ones(d,1)
# residual
R = grad_u[:,-1:] - alpha * laplace_tot + torch.matmul(grad_u[:,:-1], v) + b * u

loss = R**2
loss = torch.mean(loss)
loss.backward()
print(f"X.grad = {X.grad}")


d = 2
X.grad = tensor([[17.8629, 17.6930,  6.8999],
        [ 0.7341,  3.2902,  0.5432],
        [ 0.7423,  0.9699,  0.0557],
        [-2.7546, -0.9142, -2.3536]])


In [28]:
X = sample_domain()
X.requires_grad_(True)
# SGSD loss
N,D = X.shape
d = D-1
print(f"d = {d}")
u, grad_u, spatial_laplace_u = compute_derivatives(model, X)
laplace_tot = torch.zeros_like(u)
for i in range(d):
    laplace_tot += spatial_laplace_u[:,i:i+1]
# params
alpha = 1.0
b = 1.0
v = torch.ones(d,1)
# residual
R = grad_u[:,-1:] - alpha * laplace_tot + torch.matmul(grad_u[:,:-1], v) + b * u

# sample some indices
I = [0,1]
R_stoch = torch.zeros((N,1))
for i in I:
    Ri = 1/d * grad_u[:,-1:] - alpha * spatial_laplace_u[:,i:i+1] + v[i] * grad_u[:,i:i+1] + 1/d * b * u
    R_stoch += Ri
# total loss
loss = 2 * R.detach() * R_stoch
loss = torch.mean(loss)
loss.backward()
print(f"X.grad = {X.grad}")

d = 2
X.grad = tensor([[17.8629, 17.6930,  6.8999],
        [ 0.7341,  3.2902,  0.5432],
        [ 0.7423,  0.9699,  0.0557],
        [-2.7546, -0.9142, -2.3536]])


In [ ]:
def sdgd_loss(X, pde_sgsd_single_term_residual=None, num_dims_to_use=10):
    # sample some indices
    N,D = X.shape
    d = D-1
    I = torch.randperm(d)[:num_dims_to_use]
    u, grad_u, spatial_laplace_u = compute_derivatives(model, X)
    R_stoch = torch.zeros((N,1))
    R_stoch = grad_u[:,-1:] + b * u
    for i in I:
        #Ri = pde_sgsd_single_term_residual(X, u, grad_u, spatial_laplace_u)
        #Ri = 1/d * grad_u[:,-1:] - alpha * spatial_laplace_u[:,i:i+1] + v[i] * grad_u[:,i:i+1] + 1/d * b * u
        Ri = - alpha * spatial_laplace_u[:,i:i+1] + v[i] * grad_u[:,i:i+1]
        R_stoch += Ri
    # total loss
    loss = 2 * R.detach() * R_stoch
    loss = torch.mean(loss)
    # then do:
    #loss.backward()
    return loss

In [36]:
sdgd_loss(X, num_dims_to_use=2)

tensor(5.9506, grad_fn=<MeanBackward0>)

In [ ]:
#def sdgd_loss():
#
#    terms = pde_terms(model, x_colloc)  # All detached? No, compute with graph
#    total_res = sum(terms)  # (N,1); graph active
#    total_res_det = total_res.detach()  # Detach scalar-like res for scaling
#    
#    # Sample indices
#    N_L = len(terms)
#    I = torch.randperm(N_L)[:num_terms_sample]
#    
#    # Gradients on sampled terms only
#    grad_terms = []
#    for i in I:
#        term_i = terms[i]  # Graph active
#        g_term = torch.autograd.grad(term_i.sum(), model.parameters(), 
#                               create_graph=True, retain_graph=True)
#        grad_terms.append(g_term)
#    
#    # Aggregate: (N_L / |I|) * total_res * sum_I partial_theta L_i
#    # In practice, PyTorch autograd handles via scaled loss
#    loss_f = (N_L / num_terms_sample) * (total_res_det ** 2).mean()
#    
#    # Backward: only sampled grads flow (via retain_graph=False after)
#    loss_f.backward()
#    
#    return loss_f.item()
#
#